# Lab 2: The Plan-and-Execute Newsroom

Build a multi-agent system that uses a Planner to decompose tasks and specialist agents (researcher, analyst, writer) to execute them. Uses `@observe` from Lab 1 for end-to-end observability.

## Phase 0: Setup and Imports

In [ ]:
import requests
import json
import os
import asyncio
import logging
from typing import List
from bs4 import BeautifulSoup
from litellm import acompletion
from pydantic import BaseModel, Field
from deepeval.tracing import observe
import nest_asyncio

nest_asyncio.apply()
from dotenv import load_dotenv
load_dotenv()

MODEL_NAME = os.getenv('MODEL_NAME', 'openrouter/nvidia/nemotron-3-super-120b-a12b:free')
print(f'Environment loaded. Model: {MODEL_NAME}')

### Tool Belt

Pre-built tools for live web search.

Production tool failures are normal — your harness's job is to keep the agent moving. `search_web` below is **fail-soft**: on any failure (HTTP error, parse error, empty results) it returns a structured "no findings" marker instead of raising. The Researcher prompt in Phase 1 recognises that marker and says "no findings available" rather than inventing facts.

Same pattern as the capstone's `project_starter/src/tools/search_tool.py:search_web`. The lab and the project use the same playbook so what you build here transfers.


In [ ]:
import logging

# Module-level logger so search_web can log without depending on NewsroomAgent
# being instantiated yet. (NewsroomAgent calls basicConfig in __init__, but
# search_web may be exercised earlier during dev.)
newsroom_logger = logging.getLogger('Newsroom')


def search_web(query, max_results=3):
    """DuckDuckGo HTML search with fail-soft semantics.

    Mirrors project_starter/src/tools/search_tool.py:search_web — on any
    failure (HTTP, parse, empty results) returns a JSON list with one
    structured entry so the Researcher prompt can recognise "no findings"
    instead of hallucinating around it. Single-shot; no retries (a flaky
    DuckDuckGo turn rarely recovers on immediate retry).
    """
    url = 'https://html.duckduckgo.com/html/'
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        resp = requests.post(url, data={'q': query}, headers=headers, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')
        results = []
        for res in soup.find_all('div', class_='result', limit=max_results):
            title_tag = res.find('a', class_='result__a')
            snippet_tag = res.find('a', class_='result__snippet')
            if title_tag and snippet_tag:
                results.append({
                    'title': title_tag.get_text(strip=True),
                    'link': title_tag['href'],
                    'snippet': snippet_tag.get_text(strip=True),
                })
        if not results:
            newsroom_logger.warning(f'search_web: no results for {query!r}')
            return json.dumps([])
        return json.dumps(results)
    except Exception as e:
        newsroom_logger.error(f'search_web: request failed for {query!r}: {e}')
        return json.dumps([{'title': 'Error', 'link': '', 'snippet': f'Search failed: {e}'}])


print('Tool Belt loaded (fail-soft search_web).')


## Phase 1: Specialist Prompts

Define the system prompts for each specialist. The **Researcher** has access to `search_web`.

In [ ]:
SPECIALIST_PROMPTS = {
    'researcher': (
        'You are a Research Specialist. Use the search_web tool to find real-time information and cite your sources. '
        'If search_web returns an empty list `[]` or a single entry with title=="Error", '
        'the tool failed — say "no findings available for this query" and continue. Never invent facts to fill the gap.'
    ),
    'analyst': 'You are an Analysis Specialist. Evaluate the research, flag contradictions, and rate your confidence.',
    'writer': 'You are a Writing Specialist. Synthesize the analysis into a polished report with clear headings.',
}

SEARCH_TOOL_SCHEMA = [{
    'type': 'function',
    'function': {
        'name': 'search_web',
        'description': 'Search the web for real-time information.',
        'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']}
    }
}]

@observe(type='tool')
async def call_specialist(specialist, task):
    prompt = SPECIALIST_PROMPTS.get(specialist, 'You are a helpful assistant.')
    messages = [
        {'role': 'system', 'content': prompt},
        {'role': 'user', 'content': task}
    ]
    tools = SEARCH_TOOL_SCHEMA if specialist == 'researcher' else None
    resp = await acompletion(model=MODEL_NAME, messages=messages, tools=tools)
    msg = resp.choices[0].message
    if specialist == 'researcher' and hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            if tc.function.name == 'search_web':
                args = json.loads(tc.function.arguments)
                result = search_web(args['query'])
                messages.append(msg.model_dump())
                messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
                final = await acompletion(model=MODEL_NAME, messages=messages)
                return final.choices[0].message.content
    return msg.content

print('Specialist system ready.')


## Phase 2: The Planner

The Planner decomposes a query into ordered specialist steps using structured output.

In [ ]:
class PlanStep(BaseModel):
    step: int = Field(..., description='Step number')
    task: str = Field(..., description='Task to perform')
    specialist: str = Field(..., description='One of: researcher, analyst, writer')
    depends_on: List[int] = Field(default_factory=list)

class Plan(BaseModel):
    steps: List[PlanStep]

PLANNER_PROMPT = '''You are a task planner for a newsroom.
Specialists: researcher (finds facts), analyst (evaluates), writer (synthesizes report).
Decompose the query into minimal ordered steps. researcher steps can run independently.
Query: {query}'''

class TaskPlanner:
    @observe(type='agent')
    async def create_plan(self, query):
        resp = await acompletion(
            model=MODEL_NAME,
            messages=[{'role': 'user', 'content': PLANNER_PROMPT.format(query=query)}],
            response_format=Plan
        )
        plan = Plan.model_validate_json(resp.choices[0].message.content)
        return [s.model_dump() for s in plan.steps]

print('TaskPlanner ready.')

## Phase 3: The Newsroom Agent

Orchestrates planning and execution. Results from one step feed into the next as context.

In [ ]:
class NewsroomAgent:
    def __init__(self):
        self.planner = TaskPlanner()
        self.logger = logging.getLogger('Newsroom')
        logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', datefmt='%H:%M:%S')

    def _get_context(self, step, results):
        parts = [f'[Step {d} result]:\n{results[d]}' for d in step.get('depends_on', []) if d in results]
        return '\n\n'.join(parts)

    @observe(type='agent')
    async def run(self, query):
        self.logger.info('Planning...')
        plan = await self.planner.create_plan(query)
        for s in plan:
            self.logger.info(f'  Step {s["step"]} [{s["specialist"]}]: {s["task"]}')

        self.logger.info('Executing...')
        results = {}
        for step in plan:
            context = self._get_context(step, results)
            full_task = step['task'] if not context else f'{step["task"]}\n\nContext:\n{context}'
            self.logger.info(f'Running Step {step["step"]} [{step["specialist"]}]...')
            results[step['step']] = await call_specialist(step['specialist'], full_task)

        self.logger.info('Synthesizing...')
        summary = '\n'.join([f'Step {k}: {v}' for k, v in sorted(results.items())])
        final_resp = await acompletion(
            model=MODEL_NAME,
            messages=[
                {'role': 'system', 'content': 'Write a final cited report from these research findings.'},
                {'role': 'user', 'content': f'Query: {query}\n\nFindings:\n{summary}'}
            ]
        )
        return final_resp.choices[0].message.content

print('NewsroomAgent ready.')

## Phase 4: The Quality Gate

The cascade failure mode from Session 01 §D: the Researcher misreads a source, the Writer confidently builds a polished narrative around the wrong numbers. *Each step is locally correct — and the pipeline still ships a wrong answer.*

The fix is a **second specialist re-reading the draft against the original research findings**, with fresh context and no investment in the prose. That's the Analyst's real job in this pattern.

```
Research findings ─┐
                   ├──> Analyst (fact-checker)
Writer draft ──────┘         │
                             ├──> APPROVED  → ship
                             └──> revision notes → Writer revises
```

We upgrade the agent below in four pieces:

1. A `ReviewVerdict` model so the Analyst returns *structured* approval, not prose.
2. **`factuality_against_findings` from `shared/eval_kit/`** — the same judge primitive you saw in Module 01 Lab 04 (Arabic civics) and Module 02b Lab 02 (memory A/B). One growing artifact threaded through three modules.
3. `writer_revise` with a revision prompt that addresses the Analyst's feedback against the original findings.
4. A `_quality_gate(draft, findings, max_revisions=2)` loop, mirroring the slide pseudocode at §D.

> *Why one shared rubric instead of the four explicit checks?* The shared metric asks the narrowest grounded question — "is every claim in the candidate supported by the findings?" — which is the load-bearing one. The other concerns from §D (no upgraded confidence, no dropped findings) are exactly the case for *instantiating more LLMJudge objects*, per the rule in `eval_kit/judge.py`'s docstring: keep rubrics narrow, stack judges not prompts.

We also wrap `call_specialist` with a one-retry safety net — the gate makes the agent *more* sensitive to flaky sub-calls (one failure inside the loop would abort the whole revision).


In [ ]:
import sys
from pathlib import Path
from typing import Optional

# Wire in the shared eval_kit primitive. Same `Path.cwd().parents[1] / 'shared'`
# pattern used by Module 01 Lab 04 and Module 02b Lab 02 — see
# shared/eval_kit/README.md for the threading story.
sys.path.insert(0, str(Path.cwd().parents[1] / 'shared'))
from eval_kit.metrics import factuality_against_findings
from deepeval.test_case import LLMTestCase


class ReviewVerdict(BaseModel):
    approved: bool = Field(..., description='True if every claim in the draft is grounded in the findings.')
    feedback: str = Field(..., description='Specific revision notes when approved=False; empty when approved=True.')


WRITER_REVISE_PROMPT = '''You are the Writer. Revise your previous draft using the Analyst's feedback.
Address every point. Do not introduce new claims that are not in the original research findings.'''


# Instantiate the factuality metric once and reuse across revisions —
# matches the LLMJudge "configure once, call many" pattern in eval_kit/judge.py.
_FACTUALITY = factuality_against_findings(threshold=0.7)


async def safe_call_specialist(specialist: str, task: str, retries: int = 1) -> str:
    '''Wraps call_specialist with one retry. A single transient failure inside a
    revision loop would otherwise abort the entire gate.'''
    last_err: Optional[Exception] = None
    for attempt in range(retries + 1):
        try:
            return await call_specialist(specialist, task)
        except Exception as e:
            last_err = e
    return f'[specialist={specialist} failed after {retries + 1} attempts: {last_err}]'


@observe(type='agent')
async def analyst_review(draft: str, findings: str) -> ReviewVerdict:
    '''Fact-check the draft against the original findings using the shared
    factuality metric — the same judge primitive students saw in Module 01
    Lab 04 and Module 02b Lab 02.'''
    case = LLMTestCase(input='factuality check', actual_output=draft, context=[findings])
    await _FACTUALITY.a_measure(case)
    return ReviewVerdict(
        approved=_FACTUALITY.is_successful(),
        feedback=_FACTUALITY.reason or 'No specific issues raised by the judge.',
    )


@observe(type='agent')
async def writer_revise(draft: str, feedback: str, findings: str) -> str:
    resp = await acompletion(
        model=MODEL_NAME,
        messages=[
            {'role': 'system', 'content': WRITER_REVISE_PROMPT},
            {'role': 'user', 'content': f'## Original research findings\n{findings}\n\n## Previous draft\n{draft}\n\n## Analyst feedback\n{feedback}'},
        ],
    )
    return resp.choices[0].message.content


@observe(type='agent')
async def _quality_gate(draft: str, findings: str, max_revisions: int = 2) -> str:
    '''Slide §D pseudocode, made real. Always set max_revisions.'''
    for revision in range(max_revisions):
        verdict = await analyst_review(draft, findings)
        if verdict.approved:
            logging.getLogger('Newsroom').info(f'  gate: APPROVED on revision {revision}')
            return draft
        logging.getLogger('Newsroom').info(f'  gate: REJECTED (rev {revision}); feedback: {verdict.feedback[:120]}...')
        draft = await writer_revise(draft, verdict.feedback, findings)
    logging.getLogger('Newsroom').info(f'  gate: ship after {max_revisions} revisions (cap reached)')
    return draft


# Patch the agent: re-run with the gate wrapping the synthesis step.
@observe(type='agent')
async def run_with_gate(self, query: str) -> str:
    self.logger.info('Planning...')
    plan = await self.planner.create_plan(query)
    for s in plan:
        self.logger.info(f'  Step {s["step"]} [{s["specialist"]}]: {s["task"]}')

    self.logger.info('Executing...')
    results = {}
    for step in plan:
        context = self._get_context(step, results)
        full_task = step['task'] if not context else f'{step["task"]}\n\nContext:\n{context}'
        self.logger.info(f'Running Step {step["step"]} [{step["specialist"]}]...')
        results[step['step']] = await safe_call_specialist(step['specialist'], full_task)

    # Findings = everything the Researcher produced.
    research_findings = '\n\n'.join(
        f'[Step {s["step"]}]: {results[s["step"]]}'
        for s in plan if s['specialist'] == 'researcher' and s['step'] in results
    ) or '\n'.join(f'Step {k}: {v}' for k, v in sorted(results.items()))

    self.logger.info('Drafting...')
    draft_resp = await acompletion(
        model=MODEL_NAME,
        messages=[
            {'role': 'system', 'content': 'Write a final cited report from these research findings.'},
            {'role': 'user', 'content': f'Query: {query}\n\nFindings:\n{research_findings}'},
        ],
    )
    draft = draft_resp.choices[0].message.content

    self.logger.info('Quality gate...')
    return await _quality_gate(draft, research_findings, max_revisions=2)


NewsroomAgent.run = run_with_gate
print('NewsroomAgent upgraded with Phase 4 quality gate (using shared eval_kit.factuality_against_findings).')


## Phase 5: Run the Agent

Change `query` to any research topic and run!

In [ ]:
async def main():
    agent = NewsroomAgent()
    query = 'Compare AI regulations in the EU and Saudi Arabia'
    report = await agent.run(query)
    print(f'\n--- FINAL REPORT ---\n{report}')

await main()